# IgLM -- Antibody Language Model

![IgLM -- Antibody Language Model](https://proto-bio.github.io/proto-assets/images/tool/iglm/hero.png)

IgLM is a GPT-2 style antibody language model from the Gray Lab at Johns Hopkins, trained on antibody sequences from the Observed Antibody Space (OAS) with an *infilling* objective and conditioned on two control tokens, chain type (`heavy` / `light`) and species-of-origin (`human`, `mouse`, `rat`, `rabbit`, `rhesus`, `camel`). Because it learned to fill in a masked span rather than only to continue a prefix, one model covers three jobs: generating full-length antibodies from an N-terminal prompt (`run_iglm_sample`), re-designing an interior span such as CDR H3 while leaving the framework intact (`run_iglm_infill`), and scoring how antibody-like a whole sequence or a single designed span is (`run_iglm_score`) -- the design-then-filter loop this notebook walks through end to end.

**Links:** [Paper -- Shuai, Ruffolo & Gray, *Cell Systems* (2023)](https://doi.org/10.1016/j.cels.2023.10.001) | [GitHub](https://github.com/Graylab/IgLM)

> **License:** IgLM is released under the JHU Academic Software License Agreement -- **non-commercial use only**. Review the license before using the model or its weights in a commercial setting.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("iglm")
display_overview("iglm")
display_docs_section("iglm", "Background")

## Available tools

In [ ]:
display_available_tools("iglm")

## Imports

In [ ]:
from proto_tools.tools import (
    IgLMInfillConfig,
    IgLMInfillInput,
    IgLMSampleConfig,
    IgLMSampleInput,
    IgLMScoringConfig,
    IgLMScoringInput,
    IgLMSpan,
    run_iglm_infill,
    run_iglm_sample,
    run_iglm_score,
)

## API Reference

### `run_iglm_sample`

Generates full-length antibody sequences by continuing an N-terminal prompt from the chain and species conditioning tokens. `prompts`, `chains`, and `species` are parallel, broadcastable fields: pass one chain/species to condition every prompt the same way, or one value per prompt to mix chains and species in a single call.

In [ ]:
display_api_reference("iglm", "input", "run_iglm_sample")

In [ ]:
display_api_reference("iglm", "config", "run_iglm_sample")

In [ ]:
display_api_reference("iglm", "output", "run_iglm_sample")

### `run_iglm_infill`

Replaces a residue span with a mask token and lets IgLM write a new segment conditioned on the surrounding residues -- the objective it was trained on. The new segment is **not** constrained to the original span length, so each returned bundle reports `infilled_residues` (what IgLM wrote) alongside `spans` (where those residues sit in the returned sequence), ready to hand straight back to `run_iglm_score`. Spans are 1-indexed and inclusive, and each parent sequence needs exactly one span.

In [ ]:
display_api_reference("iglm", "input", "run_iglm_infill")
display_api_reference("iglm", "config", "run_iglm_infill")
display_api_reference("iglm", "output", "run_iglm_infill")

### `run_iglm_score`

Computes the autoregressive likelihood of each sequence under its chain and species conditioning. Supplying a `span` switches to IgLM's infilling likelihood: the span is scored given the rest of the chain, which is how designed CDR loops are ranked against each other. Omitting `spans` (or passing `None`) scores the full sequence.

In [ ]:
display_api_reference("iglm", "input", "run_iglm_score")
display_api_reference("iglm", "config", "run_iglm_score")
display_api_reference("iglm", "output", "run_iglm_score")

## 1. Generate full-length antibody sequences

IgLM resamples until it holds `num_sequences_per_input` *distinct* sequences per prompt, discarding malformed generations the same way the upstream implementation does. Generation stops at IgLM's end-of-sequence token; a candidate that instead runs out of `max_length` is discarded and charged against the attempt budget, so too tight a budget surfaces as a "yielded only N of M" error rather than as short output, which counts the two conditioning tokens and the prompt.

In [ ]:
# Five distinct human heavy chains continuing the N-terminal prompt "EVQ".
# One chain/species value conditions every prompt the same way (broadcasting).
sample_inputs = IgLMSampleInput(prompts=["EVQ"], chains=["heavy"], species=["human"])

sample_config = IgLMSampleConfig(
    model_checkpoint="IgLM",      # 13M-parameter model from the paper ("IgLM-S" is 1.4M)
    num_sequences_per_input=5,    # Distinct sequences per prompt
    temperature=1.0,
    top_p=0.95,
    prepend_prompt=True,          # Keep "EVQ" at the start of each sequence
    batch_size=8,
    seed=0,                       # Reproducible sampling
    device="cuda",                # Change to "cpu" if no GPU available
)

sample_result = run_iglm_sample(sample_inputs, sample_config)

# One IgLMSamples bundle per prompt; each bundle is iterable and indexable.
heavy_bundle = sample_result.results[0]
print(f"Generated {len(heavy_bundle)} distinct human heavy chains from the prompt 'EVQ':")
for i, sequence in enumerate(heavy_bundle, start=1):
    print(f"  {i}. ({len(sequence)} aa) {sequence}")

### Nanobodies: the same call with camel conditioning

Swapping the species token to `camel` biases generation toward camelid single-domain heavy chains (VHHs / nanobodies), which conventionally start `QVQ` rather than `EVQ`.

In [ ]:
# Camel heavy chains (nanobodies) from the prompt "QVQ".
nanobody_inputs = IgLMSampleInput(prompts=["QVQ"], chains=["heavy"], species=["camel"])
nanobody_config = IgLMSampleConfig(
    num_sequences_per_input=3,
    temperature=1.0,
    top_p=0.95,
    seed=0,
    device="cuda",  # Change to "cpu" if no GPU available
)

nanobody_result = run_iglm_sample(nanobody_inputs, nanobody_config)

print("Camel heavy chains (nanobodies) from the prompt 'QVQ':")
for i, sequence in enumerate(nanobody_result.results[0], start=1):
    print(f"  {i}. ({len(sequence)} aa) {sequence}")

## 2. Re-design a CDR loop by infilling

The usual engineering task is not generating a whole new antibody but diversifying one loop of an existing one. Here we mask CDR H3 of the 1JPT heavy chain (residues 99-106, the loop `DTAAYFDY`) and let IgLM propose replacements conditioned on the untouched framework.

In [ ]:
# 1JPT heavy chain; residues 99-106 are its CDR H3 loop.
parent_vh = (
    "EVQLVESGGGLVQPGGSLRLSCAASGFNIKEYYMHWVRQAPGKGLEWVGLIDPEQGNTIYDPKFQDRATIS"
    "ADNSKNTAYLQMNSLRAEDTAVYYCARDTAAYFDYWGQGTLVTVS"
)

# Spans are 1-indexed and inclusive, so slicing back out needs start - 1.
cdr_h3 = IgLMSpan(start=99, end=106)
original_loop = parent_vh[cdr_h3.start - 1 : cdr_h3.end]
print(f"Parent heavy chain: {len(parent_vh)} aa")
print(f"Original CDR H3:    {original_loop} ({len(original_loop)} aa)")

In [ ]:
# One span per parent sequence; chain/species broadcast as in generation.
infill_inputs = IgLMInfillInput(
    sequences=[parent_vh],
    spans=[cdr_h3],
    chains=["heavy"],
    species=["human"],
)

infill_config = IgLMInfillConfig(
    num_sequences_per_input=6,  # Six distinct re-designs of the loop
    temperature=1.0,
    top_p=0.95,
    batch_size=8,
    seed=0,
    device="cuda",  # Change to "cpu" if no GPU available
)

infill_result = run_iglm_infill(infill_inputs, infill_config)

# The bundle carries three aligned lists: the full re-designed chains, the residues
# IgLM wrote, and the span each new segment occupies in its returned sequence.
designs = infill_result.results[0]

print(f"Original CDR H3: {original_loop} ({len(original_loop)} aa)")
print("Designed CDR H3 loops (IgLM chooses the length, so they need not match):")
for i, (loop, span, design) in enumerate(
    zip(designs.infilled_residues, designs.spans, designs.sequences), start=1
):
    print(f"  design {i}: {loop:<16} ({len(loop)} aa at {span.start}-{span.end}, full chain {len(design)} aa)")

## 3. Score and rank the designs

This is the payoff of the generate-then-filter loop. We first score the unmodified parent as a
whole chain, to show what whole-sequence scoring looks like, and then re-score every design
*conditioned on its span*: IgLM reports the likelihood of just the new loop given the rest of the
chain, which is the quantity to rank CDR designs by. The two numbers are not comparable with each
other - one covers 117 tokens and the others cover a loop apiece - so the wild-type loop is carried
into the span-conditional call below to serve as the baseline for the ranking. Because the infill
output already reports `spans`, the designs hand straight back to `run_iglm_score` with no index
arithmetic.

In [ ]:
# Whole-sequence scoring: no spans, so the full chain is scored.
parent_score_inputs = IgLMScoringInput(
    sequences=[parent_vh],
    chains=["heavy"],
    species=["human"],
)
parent_score_result = run_iglm_score(
    parent_score_inputs,
    IgLMScoringConfig(batch_size=8, device="cuda"),  # Change device to "cpu" if no GPU available
)

parent_metrics = parent_score_result.scores[0]
print("Parent 1JPT heavy chain, scored as a human heavy chain:")
print(f"    Log-likelihood:     {parent_metrics.log_likelihood:.3f}")
print(f"    Avg log-likelihood: {parent_metrics.avg_log_likelihood:.3f}")
print(f"    Perplexity:         {parent_metrics.perplexity:.3f}  (lower = more antibody-like)")

In [ ]:
# Span-conditional scoring: pass the designs' own spans straight through.
# Scoring the wild-type loop in the same call gives the designs a baseline.
candidate_sequences = [parent_vh, *designs.sequences]
candidate_spans = [cdr_h3, *designs.spans]
candidate_loops = [original_loop, *designs.infilled_residues]
candidate_labels = ["wild type", *(f"design {i}" for i in range(1, len(designs.sequences) + 1))]

rank_inputs = IgLMScoringInput(
    sequences=candidate_sequences,
    spans=candidate_spans,
    chains=["heavy"],
    species=["human"],
)
rank_result = run_iglm_score(
    rank_inputs,
    IgLMScoringConfig(batch_size=8, device="cuda"),  # Change device to "cpu" if no GPU available
)

In [ ]:
# Rank by avg_log_likelihood (and its monotone partner, perplexity): the loops
# differ in length, so the summed log_likelihood is not comparable across candidates.
ranked = sorted(
    zip(candidate_labels, candidate_loops, rank_result.scores),
    key=lambda candidate: candidate[2].avg_log_likelihood,
    reverse=True,
)

print(f"{'rank':<6}{'candidate':<12}{'CDR H3':<18}{'avg log-lik':>12}{'perplexity':>13}")
for rank, (label, loop, score) in enumerate(ranked, start=1):
    print(f"{rank:<6}{label:<12}{loop:<18}{score.avg_log_likelihood:>12.3f}{score.perplexity:>13.3f}")

best_label, best_loop, best_score = ranked[0]
print(f"\nBest-scoring loop: {best_loop} ({best_label}), perplexity {best_score.perplexity:.3f}")
print("Scores are only comparable under identical conditioning (same chain and species tokens).")

## 4. Export results

Generation and infilling results export to `fasta`, `txt`, or `json`; scoring results export to `csv` or `json`.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generated sequences -> FASTA
sample_result.export(name="iglm_heavy_chains", export_path=output_dir, file_format="fasta")
print(f"Generated heavy chains exported to {output_dir / 'iglm_heavy_chains.fasta'}")

infill_result.export(name="iglm_cdr_h3_designs", export_path=output_dir, file_format="fasta")
print(f"CDR H3 re-designs exported to {output_dir / 'iglm_cdr_h3_designs.fasta'}")

# Scoring metrics -> CSV
rank_result.export(name="iglm_design_scores", export_path=output_dir, file_format="csv")
print(f"Design scores exported to {output_dir / 'iglm_design_scores.csv'}")